# LangChain Chat Models

**Main topics**
- What is a model?
- How to create a model
- `invoke()`, `stream()`, and `batch()`
- Model parameters
- Tool calling
- Structured output
- Multimodal models
- Reasoning
- Local models
- Prompt caching
- Server-side tools
- Model profiles
- Common errors



## 1. What is a model?

An LLM (Large Language Model) is an AI model that can understand and generate text.

It can be used for:
- Writing text
- Translation
- Summarization
- Answering questions
- Classification
- Extracting information

In LangChain, models are also used as the main "thinking" part of an agent.

Different models are good at different things. Some are better at following instructions, some at reasoning, and some can work with large amounts of information.


## 2. Creating a chat model

LangChain provides a common interface, so the code can be similar even when using different providers.

The documentation shows providers such as OpenAI, Anthropic, Azure, Google Gemini, AWS Bedrock, HuggingFace, and OpenRouter.

The easiest general approach is `init_chat_model()`.


In [ ]:
# Install the provider package you need.
# Example for OpenAI:
# !pip install -U "langchain[openai]"

import os
from langchain.chat_models import init_chat_model

# Store your real API key outside the notebook.
# Example:
# os.environ["OPENAI_API_KEY"] = "your-key-here"

# Example model setup:
# model = init_chat_model("gpt-5.5")

print("Model setup example is ready.")


## 3. `invoke()` — get one complete answer

`invoke()` sends a request to the model and waits for the complete answer.

Simple idea:

**Question → Model → Complete answer**

It can also receive a list of messages, which lets the model understand conversation history.


In [ ]:
# Example:
# response = model.invoke("What is machine learning?")
# print(response)

print("invoke() returns the model's complete response.")


In [ ]:
# Example with conversation history:

conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is Python?"},
]

# response = model.invoke(conversation)
# print(response)

print("A conversation can contain system, user, and assistant messages.")


## 4. `stream()` — show the answer as it is created

`stream()` gives the answer piece by piece.

This is useful when the answer is long because the user can start reading immediately instead of waiting for the whole answer.

Simple idea:

**Question → Model → small pieces → screen**


In [ ]:
# Example:
# for chunk in model.stream("Explain AI in simple words."):
#     print(chunk.text, end="")

print("stream() produces output progressively.")


## 5. `batch()` — process many questions

`batch()` is useful when you have several independent requests.

Instead of processing each request one by one, LangChain can process them in parallel.

This can improve speed and reduce waiting time.

`batch_as_completed()` can give results as each request finishes, so results may arrive in a different order.


In [ ]:
questions = [
    "What is AI?",
    "What is Python?",
    "What is a database?"
]

# responses = model.batch(questions)
# for response in responses:
#     print(response)

print("batch() is useful for multiple independent requests.")


## 6. Important model parameters

A model can have settings that change how it behaves.

### `model`
The name of the model you want to use.

### `api_key`
The secret key used to connect to the provider.

### `temperature`
Controls randomness.
- Higher value → more creative/random
- Lower value → more predictable

### `max_tokens`
Controls the maximum size of the generated answer.

### `timeout`
How many seconds to wait before giving up.

### `max_retries`
How many times LangChain can retry after certain failures such as network problems or rate limits.


In [ ]:
# Example settings:

# model = init_chat_model(
#     "claude-sonnet-4-6",
#     temperature=0.7,
#     timeout=30,
#     max_tokens=1000,
#     max_retries=6
# )

print("These settings control model behavior and reliability.")


## 7. Tool calling

Tool calling means the model can ask a program to perform a task.

Examples:
- Search the web
- Get weather data
- Query a database
- Run code

The basic flow is:

**User asks → Model decides a tool is needed → Tool runs → Result goes back to model → Model gives final answer**

In LangChain, tools are connected to a model using `bind_tools()`.


In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get simple weather information for a location."""
    return f"The weather in {location} is sunny."

# model_with_tools = model.bind_tools([get_weather])

print("A Python function can be turned into a tool with @tool.")


## 8. Structured output

Normally, a model returns text.

Sometimes an application needs a fixed format, such as:

- title
- year
- director
- rating

Structured output tells the model to return data that follows a defined schema.

LangChain supports:
- Pydantic
- TypedDict
- JSON Schema

This is useful when another program needs to read the model's answer reliably.


In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="Movie title")
    year: int = Field(description="Release year")
    director: str = Field(description="Director name")
    rating: float = Field(description="Rating out of 10")

# structured_model = model.with_structured_output(Movie)
# response = structured_model.invoke("Give details about Inception")
# print(response)

print("Pydantic can describe the structure we want from the model.")


## 9. Multimodal models

Some models can work with more than text.

They may support:
- Images
- Audio
- Video

LangChain can pass this information using content blocks.

So a model can sometimes understand an image and return text or another supported type of content.


## 10. Reasoning

Some models can perform multi-step reasoning.

LangChain can expose reasoning information when the underlying model supports it.

Some providers also allow a reasoning effort setting, such as low, medium, or high.

The exact supported levels depend on the model/provider.


In [ ]:
# Example shown in the documentation:

# response = model.invoke(
#     "Why do parrots have colorful feathers?",
#     reasoning_effort="high",
# )

print("Reasoning settings depend on the model and provider.")


## 11. Local models

LangChain can also work with models running on your own computer.

This can be useful when:
- Data privacy is important
- You want to use a custom model
- You want to avoid cloud model costs

The documentation mentions Ollama as an easy option for running models locally.


## 12. Prompt caching

Prompt caching can reduce repeated work.

If the same prompt information is sent many times, a provider may reuse cached information.

Benefits can include:
- Lower waiting time
- Lower cost

Caching can be provided by the model provider or by LangChain middleware.

The exact behavior depends on the provider.


## 13. Server-side tools

Some providers can run tools on their own servers.

For example, a model may use a web-search tool and process the search result as part of one conversation turn.

This is different from client-side tool calling, where your application receives the tool request and runs the tool itself.


## 14. Model profiles

A model can have a `profile` that describes what it supports.

For example, a profile may contain information about:
- Maximum input tokens
- Image support
- Reasoning support
- Tool calling
- Structured output

Applications can use this information to choose features that the current model supports.


In [ ]:
# Example:
# print(model.profile)

print("A model profile describes the capabilities of a model.")


## 15. Error handling

Model calls can fail because of:
- Authentication problems
- Rate limits
- Timeouts
- Network problems
- Server errors

LangChain provides standard exception types for common model failures.

It also retries some temporary failures automatically. The documentation says the default `max_retries` is 6.


## 16. Quick revision

| Feature | Simple meaning |
|---|---|
| Model | AI that understands and generates content |
| `invoke()` | Get one complete answer |
| `stream()` | Get the answer piece by piece |
| `batch()` | Process many requests |
| Tool calling | Let the model ask a program to do a task |
| Structured output | Get data in a fixed format |
| Multimodal | Work with text, images, audio, or video |
| Reasoning | Multi-step problem solving |
| Local model | Run the model on your own hardware |
| Prompt caching | Reuse repeated prompt information |
| Model profile | Information about model capabilities |
| `temperature` | Controls randomness |
| `max_tokens` | Controls response length |
| `timeout` | Maximum waiting time |
| `max_retries` | Number of retry attempts |


## Conclusion

The main idea is simple:

**LangChain gives a common way to work with many AI chat models.**

The most important things to remember are:

1. Create a model.
2. Use `invoke()` for a normal complete response.
3. Use `stream()` when you want output as it is generated.
4. Use `batch()` for many independent requests.
5. Use tools when the model needs outside actions or data.
6. Use structured output when your application needs predictable data.
7. Check model capabilities before using advanced features.
8. Keep API keys and other secrets out of GitHub.
